# Layer 06 — Property Search Demo (Neo4j)

Demonstrates the weighted property search interface backed by **Neo4j AuraDB**.

All searches use `Neo4jPropertySearch` which executes Cypher queries against the graph.
The `PropertyKnowledgeBase` (parquet) is kept as a local fallback.

## Scoring dimensions (0–10, higher = better)

| Score | Measures | Direction |
|-------|---------|----------|
| `score_famous_school` | Proximity to autonomous/gifted/SAP primary school | inverse |
| `score_school_quality` | Quality of primary schools within 1 km | direct |
| `score_school_proximity` | Distance to nearest primary school (any) | inverse |
| `score_mrt` | Distance to nearest MRT/LRT | inverse |
| `score_food` | Distance to nearest hawker/food court | inverse |
| `score_shopping` | Distance to nearest mall | inverse |
| `score_size` | Floor area (sqm) | direct |
| `score_floor` | Floor level | direct |
| `score_lease` | Remaining lease years | direct |
| `score_quietness` | Distance from highway (farther = quieter) | direct |
| `score_value` | Value for money (lower price = higher score) | inverse |
| `score_orientation` | Unit orientation | direct |

## Prerequisites

Run `01_build_knowledge_base.ipynb` to build the KB and push it to Neo4j.

## Cell 1 — Connect to Neo4j

In [2]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / 'hf_data').exists() else cwd.parent
LAYER_DIR = REPO_ROOT / '06_search_layer'

if str(LAYER_DIR) not in sys.path:
    sys.path.insert(0, str(LAYER_DIR))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

from yc_property_search import Neo4jPropertySearch

# Test connection
with Neo4jPropertySearch() as neo4j:
    counts = neo4j.node_counts()
print('Neo4j graph ready:', counts)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:.3f}'.format)

Neo4j graph ready: {'Property': 9710, 'Town': 26, 'FamousSchool': 16}


## Scenario 1 — Education priority

A family buying a flat specifically for primary school registration phase.
High weight on `score_famous_school` means the composite score is dominated by
proximity to autonomous/gifted/SAP schools.

**Expected:** All top-10 have `dist_to_nearest_famous_school_km` < 1.0 km.

In [5]:
with Neo4jPropertySearch() as neo4j:
    edu_results = neo4j.search(
        weights={
            'score_famous_school': 1,   # primary priority
            'score_school_quality': 1,   # general school quality
            'score_mrt': 10,
            'score_size': 1,
            'score_lease': 1,
        },
        filters={
            'town': 'YISHUN',       # restrict to Toa Payoh only
        },
        top_k=1000,
    )

print('=== Education Priority — Toa Payoh (Neo4j) ===')
display(edu_results[[
    'address_key', 'town', 'flat_type', 'floor_area_sqm', 'resale_price',
    'dist_to_nearest_famous_school_km', 'nearest_famous_school_name',
    'score_famous_school', 'score_mrt', 'composite_score'
]].sort_values('score_mrt', ascending=True).head(20))

max_dist = edu_results['dist_to_nearest_famous_school_km'].max()
print(f'\nMax distance to famous school in results: {max_dist:.3f} km  ({"✓ all within 1 km" if max_dist < 1.0 else "⚠ some outside 1 km"})')


=== Education Priority — Toa Payoh (Neo4j) ===


,address_key,town,flat_type,floor_area_sqm,resale_price,dist_to_nearest_famous_school_km,nearest_famous_school_name,score_famous_school,score_mrt,composite_score
630,289 YISHUN AVE 6,YISHUN,4 ROOM,84.000,460000.000,8.510,ROSYTH SCHOOL,1.455,0.559,1.528
590,286 YISHUN AVE 6,YISHUN,4 ROOM,105.000,600000.000,8.476,ROSYTH SCHOOL,1.491,0.647,1.812
627,285 YISHUN AVE 6,YISHUN,3 ROOM,64.000,335000.000,8.433,ROSYTH SCHOOL,1.536,0.657,1.535
635,288 YISHUN AVE 6,YISHUN,4 ROOM,84.000,475000.000,8.526,ROSYTH SCHOOL,1.437,0.672,1.493
628,284 YISHUN AVE 6,YISHUN,3 ROOM,74.000,400000.000,8.438,ROSYTH SCHOOL,1.530,0.706,1.530
609,287 YISHUN AVE 6,YISHUN,4 ROOM,105.000,530000.000,8.487,ROSYTH SCHOOL,1.478,0.728,1.706
624,283 YISHUN AVE 6,YISHUN,3 ROOM,74.000,416888.000,8.395,ROSYTH SCHOOL,1.576,0.737,1.558
625,282 YISHUN AVE 6,YISHUN,3 ROOM,70.000,420000.000,8.431,ROSYTH SCHOOL,1.538,0.761,1.547
505,281 YISHUN ST 22,YISHUN,EXECUTIVE,152.000,685000.000,8.438,ROSYTH SCHOOL,1.531,0.846,2.209
620,280 YISHUN ST 22,YISHUN,4 ROOM,84.000,480000.000,8.432,ROSYTH SCHOOL,1.537,0.915,1.596



Max distance to famous school in results: 8.564 km  (⚠ some outside 1 km)


## Scenario 2 — Commuter priority

In [11]:
with Neo4jPropertySearch() as neo4j:
    commuter_results = neo4j.search(
        weights={
            'score_mrt': 10,
            'score_food': 8,
            'score_shopping': 5,
            'score_value': 6,
        },
        top_k=10,
    )

print('=== Commuter Priority — Top 10 ===')
display(commuter_results[[
    'address_key', 'town', 'flat_type', 'floor_area_sqm', 'resale_price',
    'dist_to_mrt_m', 'score_mrt', 'score_food', 'composite_score'
]])
print(f'\nMean MRT distance in top-10: {commuter_results["dist_to_mrt_m"].mean():.0f} m')

=== Commuter Priority — Top 10 ===


,address_key,town,flat_type,floor_area_sqm,resale_price,dist_to_mrt_m,score_mrt,score_food,composite_score
0,441B FERNVALE RD,SENGKANG,2 ROOM,50.000,398000.000,195.141,9.967,9.648,9.569
1,418A FERNVALE LINK,SENGKANG,2 ROOM,38.000,395000.000,128.244,10.000,9.670,9.537
2,443B FERNVALE RD,SENGKANG,2 ROOM,47.000,358888.000,274.029,9.842,9.218,9.493
3,441C FERNVALE RD,SENGKANG,2 ROOM,50.000,390000.000,173.647,10.000,9.688,9.482
4,85 LOR 4 TOA PAYOH,TOA PAYOH,3 ROOM,68.000,380000.000,499.280,9.485,9.704,9.474
5,32 NEW MKT RD,CENTRAL AREA,2 ROOM,52.000,390000.000,109.387,10.000,8.630,9.440
6,85B LOR 4 TOA PAYOH,TOA PAYOH,3 ROOM,68.000,400000.000,499.847,9.484,9.707,9.409
7,91 LOR 3 TOA PAYOH,TOA PAYOH,3 ROOM,68.000,345000.000,319.617,9.769,10.000,9.405
8,453A FERNVALE RD,SENGKANG,2 ROOM,46.000,360000.000,300.171,9.800,9.226,9.403
9,56 LOR 4 TOA PAYOH,TOA PAYOH,3 ROOM,57.000,290000.000,552.311,9.400,10.000,9.382



Mean MRT distance in top-10: 305 m


## Scenario 3 — Family search with town + flat type filter

In [12]:
with Neo4jPropertySearch() as neo4j:
    family_results = neo4j.search(
        weights={
            'score_famous_school': 9,
            'score_size': 8,
            'score_quietness': 7,
            'score_lease': 6,
            'score_mrt': 5,
        },
        filters={
            'flat_type': '4 ROOM',
            'town': 'BISHAN',
        },
        top_k=10,
    )

print('=== Family Search (4-room, Bishan) — Top 10 ===')
if family_results.empty:
    print('No results — try relaxing the town filter.')
else:
    display(family_results[[
        'address_key', 'town', 'flat_type', 'floor_area_sqm', 'resale_price',
        'dist_to_nearest_famous_school_km', 'nearest_famous_school_name',
        'score_famous_school', 'score_size', 'composite_score'
    ]])

=== Family Search (4-room, Bishan) — Top 10 ===


,address_key,town,flat_type,floor_area_sqm,resale_price,dist_to_nearest_famous_school_km,nearest_famous_school_name,score_famous_school,score_size,composite_score
0,449 SIN MING AVE,BISHAN,4 ROOM,110.000,770000.000,0.373,AI TONG SCHOOL,10.000,5.610,5.917
1,446 BRIGHT HILL DR,BISHAN,4 ROOM,105.000,650000.000,0.312,AI TONG SCHOOL,10.000,5.000,5.879
2,273A BISHAN ST 24,BISHAN,4 ROOM,95.000,1250000.000,1.301,AI TONG SCHOOL,9.057,3.780,5.807
3,451 SIN MING AVE,BISHAN,4 ROOM,107.000,810000.000,0.401,AI TONG SCHOOL,10.000,5.244,5.791
4,443 SIN MING AVE,BISHAN,4 ROOM,105.000,710000.000,0.208,AI TONG SCHOOL,10.000,5.000,5.761
5,445 SIN MING AVE,BISHAN,4 ROOM,105.000,855000.000,0.254,AI TONG SCHOOL,10.000,5.000,5.744
6,441 SIN MING AVE,BISHAN,4 ROOM,103.000,818888.000,0.218,AI TONG SCHOOL,10.000,4.756,5.677
7,248 BISHAN ST 22,BISHAN,4 ROOM,116.000,950000.000,1.144,AI TONG SCHOOL,9.222,6.341,5.668
8,454 SIN MING AVE,BISHAN,4 ROOM,103.000,798000.000,0.318,AI TONG SCHOOL,10.000,4.756,5.657
9,264 BISHAN ST 24,BISHAN,4 ROOM,112.000,875000.000,1.026,AI TONG SCHOOL,9.346,5.854,5.649


## Scenario 4 — Value seeker with budget constraint

In [13]:
with Neo4jPropertySearch() as neo4j:
    value_results = neo4j.search(
        weights={
            'score_value': 10,
            'score_lease': 7,
            'score_mrt': 6,
            'score_size': 5,
        },
        filters={
            'max_resale_price': 800_000,
            'min_lease_years': 50,
        },
        top_k=10,
    )

print('=== Value Seeker (≤$800k, ≥50yr lease) — Top 10 ===')
display(value_results[[
    'address_key', 'town', 'flat_type', 'floor_area_sqm', 'resale_price',
    'lease_remaining_years', 'dist_to_mrt_m', 'score_value', 'score_lease', 'composite_score'
]])
print(f'\nPrice range: ${value_results["resale_price"].min():,.0f} – ${value_results["resale_price"].max():,.0f}')

=== Value Seeker (≤$800k, ≥50yr lease) — Top 10 ===


,address_key,town,flat_type,floor_area_sqm,resale_price,lease_remaining_years,dist_to_mrt_m,score_value,score_lease,composite_score
0,327B SUMANG WALK,PUNGGOL,2 ROOM,47.000,365000.000,92,95.874,10.000,10.000,8.214
1,222 PENDING RD,BUKIT PANJANG,5 ROOM,128.000,385000.000,69,331.438,9.840,4.762,8.188
2,405B NORTHSHORE DR,PUNGGOL,2 ROOM,47.000,380000.000,94,40.968,9.920,10.000,8.186
3,659B PUNGGOL EAST,PUNGGOL,2 ROOM,47.000,372000.000,91,317.226,10.000,10.000,8.166
4,659C PUNGGOL EAST,PUNGGOL,2 ROOM,38.000,320000.000,92,342.070,10.000,10.000,8.157
5,405A NORTHSHORE DR,PUNGGOL,2 ROOM,38.000,387000.000,94,98.047,9.808,10.000,8.146
6,421A NORTHSHORE DR,PUNGGOL,2 ROOM,38.000,375000.000,94,388.586,10.000,10.000,8.142
7,995B BUANGKOK CRES,HOUGANG,2 ROOM,47.000,365000.000,93,471.148,10.000,10.000,8.113
8,418A FERNVALE LINK,SENGKANG,2 ROOM,38.000,395000.000,91,128.244,9.680,10.000,8.100
9,547C SEGAR RD,BUKIT PANJANG,2 ROOM,47.000,355000.000,89,124.346,10.000,9.524,8.095



Price range: $320,000 – $395,000


## Scenario 5 — Hard famous school requirement + balanced lifestyle

In [14]:
with Neo4jPropertySearch() as neo4j:
    custom_results = neo4j.search(
        weights={
            'score_famous_school': 8,
            'score_mrt': 7,
            'score_food': 6,
            'score_size': 5,
            'score_quietness': 5,
            'score_value': 4,
        },
        filters={
            'require_famous_school': True,   # must be within 1 km of a famous school
            'min_floor_area': 90,
        },
        top_k=15,
    )

print('=== Custom Multi-Criteria (near famous school, ≥90sqm) — Top 15 ===')
display(custom_results[[
    'address_key', 'town', 'flat_type', 'floor_area_sqm', 'resale_price',
    'dist_to_nearest_famous_school_km', 'nearest_famous_school_name', 'composite_score'
]])
print(f'\n{len(custom_results)} properties returned (all within 1km of a famous school)')

=== Custom Multi-Criteria (near famous school, ≥90sqm) — Top 15 ===


,address_key,town,flat_type,floor_area_sqm,resale_price,dist_to_nearest_famous_school_km,nearest_famous_school_name,composite_score
0,858B TAMPINES AVE 5,TAMPINES,4 ROOM,108.000,428000.000,0.532,POI CHING SCHOOL,8.040
1,860A TAMPINES AVE 5,TAMPINES,4 ROOM,104.000,450000.000,0.450,POI CHING SCHOOL,7.873
2,903 TAMPINES AVE 4,TAMPINES,5 ROOM,139.000,835000.000,0.337,ST. HILDA'S PRIMARY SCHOOL,7.825
3,906 TAMPINES AVE 4,TAMPINES,4 ROOM,104.000,532000.000,0.339,ST. HILDA'S PRIMARY SCHOOL,7.728
4,149 TAMPINES ST 12,TAMPINES,EXECUTIVE,150.000,960000.000,0.663,ST. HILDA'S PRIMARY SCHOOL,7.720
5,913 TAMPINES ST 91,TAMPINES,4 ROOM,104.000,490000.000,0.237,ST. HILDA'S PRIMARY SCHOOL,7.708
6,856E TAMPINES ST 82,TAMPINES,EXECUTIVE,147.000,1038000.000,0.391,ST. HILDA'S PRIMARY SCHOOL,7.690
7,148 TAMPINES AVE 5,TAMPINES,EXECUTIVE,148.000,955000.000,0.591,ST. HILDA'S PRIMARY SCHOOL,7.688
8,912 TAMPINES ST 91,TAMPINES,5 ROOM,133.000,800000.000,0.250,ST. HILDA'S PRIMARY SCHOOL,7.683
9,856D TAMPINES ST 82,TAMPINES,EXECUTIVE,154.000,1188000.000,0.330,ST. HILDA'S PRIMARY SCHOOL,7.662



15 properties returned (all within 1km of a famous school)


---
## Graph traversal — Neo4j-specific queries

Beyond flat scoring, the graph model enables relationship-based queries not possible with parquet.

In [15]:
# Which towns have the most properties within 1 km of a famous school?
with Neo4jPropertySearch() as neo4j:
    result = neo4j.graph_query(
        "MATCH (p:Property)-[:NEAR_FAMOUS_SCHOOL]->(s:FamousSchool) "
        "WHERE p.famous_school_count_1km > 0 "
        "RETURN p.town AS town, count(DISTINCT p) AS properties, "
        "collect(DISTINCT s.name)[..3] AS sample_schools "
        "ORDER BY properties DESC"
    )

print('Towns with most properties within 1km of a famous school:')
display(pd.DataFrame(result))

Towns with most properties within 1km of a famous school:


,town,properties,sample_schools
0,TAMPINES,312,"[POI CHING SCHOOL, RED SWASTIKA SCHOOL, ST. HI..."
1,HOUGANG,288,"[HOLY INNOCENTS' PRIMARY SCHOOL, ROSYTH SCHOOL]"
2,BEDOK,224,"[MAHA BODHI SCHOOL, RED SWASTIKA SCHOOL, ST. H..."
3,TOA PAYOH,167,"[PEI CHUN PUBLIC SCHOOL, HONG WEN SCHOOL]"
4,KALLANG/WHAMPOA,157,"[KONG HWA SCHOOL, PEI CHUN PUBLIC SCHOOL, HONG..."
5,GEYLANG,120,"[MAHA BODHI SCHOOL, KONG HWA SCHOOL, TAO NAN S..."
6,SERANGOON,102,[ROSYTH SCHOOL]
7,CLEMENTI,96,"[PEI HWA PRESBYTERIAN PRIMARY SCHOOL, NAN HUA ..."
8,MARINE PARADE,54,[TAO NAN SCHOOL]
9,BISHAN,45,"[AI TONG SCHOOL, PEI CHUN PUBLIC SCHOOL]"


In [16]:
# All properties within 1 km of NANYANG PRIMARY SCHOOL, ranked by score_famous_school
with Neo4jPropertySearch() as neo4j:
    result = neo4j.graph_query(
        "MATCH (p:Property)-[r:NEAR_FAMOUS_SCHOOL]->(s:FamousSchool {name: $name}) "
        "WHERE r.distance_km < 1.0 "
        "RETURN p.address_key AS address_key, p.town AS town, p.flat_type AS flat_type, "
        "p.resale_price AS resale_price, r.distance_km AS distance_km, "
        "p.score_famous_school AS score_famous_school "
        "ORDER BY r.distance_km ASC "
        "LIMIT 10",
        name='NANYANG PRIMARY SCHOOL'
    )

print('Properties within 1km of NANYANG PRIMARY SCHOOL:')
display(pd.DataFrame(result))

Properties within 1km of NANYANG PRIMARY SCHOOL:


,address_key,town,flat_type,resale_price,distance_km,score_famous_school
0,4 QUEEN'S RD,BUKIT TIMAH,3 ROOM,460000.000,0.253,10.000
1,2 QUEEN'S RD,BUKIT TIMAH,4 ROOM,660000.000,0.418,9.988
2,6 FARRER RD,BUKIT TIMAH,4 ROOM,620000.000,0.419,9.986
3,1 QUEEN'S RD,BUKIT TIMAH,5 ROOM,900888.000,0.435,9.970
4,3 QUEEN'S RD,BUKIT TIMAH,4 ROOM,670000.000,0.484,9.918
5,5 FARRER RD,BUKIT TIMAH,5 ROOM,1000000.000,0.503,9.898
6,8 EMPRESS RD,BUKIT TIMAH,3 ROOM,445000.000,0.578,9.819


---
## Reference — search() interface

```python
with Neo4jPropertySearch() as neo4j:
    results = neo4j.search(
        weights={
            # Assign 0–10 to any combination; 0 = ignore that dimension
            'score_famous_school'  : 10,
            'score_school_quality' : 8,
            'score_mrt'            : 7,
            'score_food'           : 5,
            'score_shopping'       : 3,
            'score_size'           : 6,
            'score_floor'          : 2,
            'score_lease'          : 4,
            'score_quietness'      : 5,
            'score_value'          : 8,
            'score_orientation'    : 1,
        },
        filters={
            'flat_type'            : '4 ROOM',
            'town'                 : 'BISHAN',
            'min_floor_area'       : 90,
            'max_resale_price'     : 900_000,
            'min_lease_years'      : 50,
            'max_dist_mrt_m'       : 800,
            'require_famous_school': True,
        },
        top_k=20,
    )
```